<a href="https://colab.research.google.com/github/Yusuf-kadri/Yusuf-kadri---AnalyticsEngineering/blob/main/20_AEAIV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# Analytics Engineering: timed test

## **PLEASE FIRST SAVE A COPY OF THIS FILE! GO TO FILE > SAVE A COPY IN DRIVE.**
### This will mean that your work auto-saves and prevents any issues if you have internet or browser issues. Remember to work on the saved copy. It will be titled "Copy of AE_test_task ..."
<br>


## Task B: Delivery performance

## How this test works

- You have **55 minutes to complete the work** with an additional 5 minutes at the end to upload the work. However, there is no end-point as such, you are completing part of what would likely be a longer task.
- This is individual work - please do not confer with anyone inside or outside the room.
- You may use **any AI tool** (for example, Gemini in Colab - which is probably the quickest route). You are **not marked on the quality of the code** it produces. You are marked on how well you control the work: what you look at, what you check, what you reject or correct and why, and what you could not verify.
- Therefore the quality of the AI used has no bearing on your marks, as you are **not marked on the quality of the code** it produces.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make. Notes written afterwards are usually vaguer. If you are using Gemini as in built in the Notebook the prompts are **not** automatically saved - you still need to paste them into section 5 or alongside your code, **along with the checks you determine to make sure the AI code works**.
- Unfinished work is fine if your checking and reasoning are clear. Say what you did not get to.
- Please do not discuss this task with anyone who has not yet sat the test. Other groups have different briefs.

## Rubric

| Element | Marks |
| --- | --- |
| Question one | 100% |
| | |
| Application of analytics engineering techniques: applying methods from the module such as data cleaning, joining, etc. | 35% |
| Careful execution of code (whether generated by AI, from module resources (e.g. GitHub) or memory) and checking the code has worked.  | 35% |
| Analysis and interpretation of results: what do your checks show about the data and what steps should come next? | 30% |




## The data

You have eight tables from a real Brazilian online marketplace (Olist), covering orders from 2016 to 2018. A geolocation table exists in the full dataset but is not included in this test.

![](https://drive.google.com/uc?export=view&id=10zwOXJ1e9IpT58rhvxgeMqaUZXiqYzDJ)

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | The core table: one record per order, with status and timestamps |
| `order_items` | The items inside each order |
| `payments` | How each order was paid |
| `reviews` | Customer reviews of orders |
| `customers` | Customer details and location |
| `products` | Product details, including category |
| `sellers` | Seller details and location |
| `category_translation` | Maps product category names to English |

**How the tables connect**

- `orders` to `order_items`, `payments` and `reviews` on `order_id`
- `orders` to `customers` on `customer_id`
- `order_items` to `products` on `product_id`
- `order_items` to `sellers` on `seller_id`
- `category_translation` to `products` on the product category name

Check the actual column names in the next cell before you write any joins.


In [1]:
# ---- Setup: run this cell first ----
import glob
import os

import duckdb
import pandas as pd

# Download the files for the test
DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/testsession/brazilian-ecommerce.zip"    # link to a .zip containing the CSV files

DATA_DIR = "/content/olist"   # upload the CSV files here using the Files panel
  # Download only if no CSV files are there yet. The CSVs may sit inside a folder
  # within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/olist.zip")
    zipfile.ZipFile("/content/olist.zip").extractall(DATA_DIR)

FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask your invigilator for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 99,441 rows, 8 columns
    ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
order_items: 112,650 rows, 7 columns
    ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
payments: 103,886 rows, 5 columns
    ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
reviews: 89,999 rows, 7 columns
    ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
customers: 99,441 rows, 5 columns
    ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
products: 32,951 rows, 9 columns
    ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'produc

## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.


In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, c.customer_id
    FROM orders AS o
    JOIN customers AS c ON o.customer_id = c.customer_id
    LIMIT 5
"""))


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,n_rows,n_distinct_orders
0,99441,99441


,order_id,customer_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["customers"], on="customer_id", how="inner")
display(joined[["order_id", "customer_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


(99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


,order_id,customer_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c


99441 different customer_id values in orders


## Your brief

**From:** Operations

> We want to know how often orders arrive on time. Show the on-time delivery rate by **month** and by **seller state**, and tell us which **product categories** have the most late deliveries.

**What your final output should look like**

1. A table with columns for month, seller state, number of orders and on-time rate.
2. A short ranked list of the product categories with the most late deliveries.

State clearly what one row of each table represents.

**Decisions you must make and state**

- What counts as "late", and which two dates will you compare?
- How will you treat orders that were not delivered?

**If you have time**

Add the average number of days late for late orders.


## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

Strategy: The brief only requires `orders` (timestamps and status), `order_items` (to connect sellers and products), `sellers` (for seller state), `products` (for product categories), and `category_translation`. Therefore, I will examine the dataset granularity, primary/foreign keys, and the two delivery dates to be compared. Auxiliary tables like `payments`, `reviews`, and `customers` are not needed for this scope.

Prompt: For orders, order_items, sellers, products and category_translation, print the rows, the distinct values of the key and the missing values per column.

Expect: I expect to find 99,441 rows and 99,441 unique values for order_id in the orders table. The order_items table should contain more rows than orders, reflecting orders with multiple items.

In [4]:
orders = tables["orders"].copy()
items = tables["order_items"]
sellers = tables["sellers"]
products = tables["products"]
translation = tables["category_translation"]

for name, df, key in [("orders", orders, "order_id"), ("order_items", items, "order_id"), ("sellers", sellers, "seller_id"),
                      ("products", products, "product_id"), ("category_translation", translation, "product_category_name")]:
    missing = df.isna().sum()
    print(f"{name:21s} rows: {len(df):>7,} | distinct {key}: {df[key].nunique():>7,} | missing: {missing[missing > 0].to_dict() or 'none'}")

orders                rows:  99,441 | distinct order_id:  99,441 | missing: {'order_approved_at': 160, 'order_delivered_carrier_date': 1783, 'order_delivered_customer_date': 2965}
order_items           rows: 112,650 | distinct order_id:  98,666 | missing: none
sellers               rows:   3,095 | distinct seller_id:   3,095 | missing: none
products              rows:  32,951 | distinct product_id:  32,951 | missing: {'product_category_name': 610, 'product_name_lenght': 610, 'product_description_lenght': 610, 'product_photos_qty': 610, 'product_weight_g': 2, 'product_length_cm': 2, 'product_height_cm': 2, 'product_width_cm': 2}
category_translation  rows:      71 | distinct product_category_name:      71 | missing: none


Based on the initial inspection:

*   **Granularity and Keys:** The keys `order_id`, `seller_id`, and `product_id` are unique primary keys in their respective master tables. However, the `order_items` table contains 112,650 rows across only 98,666 unique orders, confirming its granularity is at the individual item level rather than the order level.
*   **Missing Order Timestamps:** There are 2,965 orders that do not contain a customer delivery date.
*   **Missing Product Categories:** There are 610 products in the catalog that do not have a designated category.

Prompt: Show me the value counts of order_status, and cross-tabulate order_status against whether order_delivered_customer_date is missing.

Expect: I expect most orders to have a status of 'delivered', and only those delivered orders should have a populated delivery date.

In [5]:
pd.crosstab(orders["order_status"], orders["order_delivered_customer_date"].isna(), margins=True)

order_delivered_customer_date,False,True,All
order_status,,,
approved,0,2,2
canceled,6,619,625
created,0,5,5
delivered,96470,8,96478
invoiced,0,314,314
processing,0,301,301
shipped,0,1107,1107
unavailable,0,609,609
All,96476,2965,99441


Saw: mostly as expected, with two operational anomalies:

1.  **Delivered orders without a date:** There are 8 orders marked as 'delivered' that are missing a customer delivery timestamp.
2.  **Canceled orders with a date:** There are 6 orders marked as 'canceled' that do have a customer delivery timestamp.

All remaining 2,957 undelivered orders are distributed across statuses such as shipped, canceled, unavailable, or in progress.

Prompt: Convert the four order timestamp columns to datetimes. it should show the range of order_purchase_timestamp, the number of orders per purchase month, and whether order_estimated_delivery_date ever has a time other than midnight.

Expect: I expect orders to span from late 2016 through mid-2018, demonstrating growing transaction volumes over time.

In [6]:
for col in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
            "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[col] = pd.to_datetime(orders[col])

print("purchases from", orders["order_purchase_timestamp"].min(), "to", orders["order_purchase_timestamp"].max())
print("estimated dates with a time other than midnight:", (orders["order_estimated_delivery_date"].dt.hour != 0).sum())
orders["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()

purchases from 2016-09-04 21:15:19 to 2018-10-17 17:30:18
estimated dates with a time other than midnight: 0


,count
order_purchase_timestamp,
2016-09,4
2016-10,324
2016-12,1
2017-01,800
2017-02,1780
2017-03,2682
2017-04,2404
2017-05,3700
2017-06,3245


Saw:

*   **Estimated Dates Lack Time Precision:** The estimated delivery column (`order_estimated_delivery_date`) contains no time components (all are recorded exactly at midnight). This implies that a delivery is considered on time as long as it arrives "by the end of that calendar day." Conversely, the actual customer delivery column includes hours and minutes. Making a direct raw timestamp comparison would incorrectly penalize any order delivered on the promised day after 00:00:00 as late. I will measure the scale of this issue in Section 3.
*   **Sparse Edge Months:** Transaction volume is extremely sparse at the boundaries: September 2016 (4 orders), December 2016 (1 order), September 2018 (16 orders), and October 2018 (4 orders), with no transactions in November 2016. Because of the tiny sample sizes, performance metrics for these boundary months will not yield reliable insights.

Prompt: For order_items, count how many orders have items from more than one seller, from more than one seller state, and from more than one product category.

Expect: I expect these numbers to be relatively small, as the vast majority of orders consist of a single item.

In [7]:
items_sellers = items.merge(sellers, on="seller_id", how="left")
items_products = items.merge(products, on="product_id", how="left")

print("orders with >1 seller:      ", (items.groupby("order_id")["seller_id"].nunique() > 1).sum())
print("orders with >1 seller state:", (items_sellers.groupby("order_id")["seller_state"].nunique() > 1).sum())
print("orders with >1 category:    ", (items_products.groupby("order_id")["product_category_name"].nunique() > 1).sum())
print("categories not in the translation table:", set(products["product_category_name"].dropna()) - set(translation["product_category_name"]))

orders with >1 seller:       1278
orders with >1 seller state: 510
orders with >1 category:     727
categories not in the translation table: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


Prompt: Based one the above understanding of the data, create a summary and decisions to be made going foreward

### **Section 1 Summary & Analytical Decisions**

#### **1. Data Structure & Relationships Summary**
| Table | One row represents | Key Insights & Data Quirks | Action / Mitigation Plan |
| :--- | :--- | :--- | :--- |
| **`orders`** | One distinct order | • 2,965 orders are missing `order_delivered_customer_date`. <br>• 8 "delivered" orders have no delivery date. <br>• 6 "canceled" orders have a delivery date. <br>• `order_estimated_delivery_date` has only midnight (`00:00:00`) timestamps. | • Establish clear operational filters for undelivered orders. <br>• Normalize dates to prevent false-positive late delivery flags. |
| **`order_items`** | One individual item inside an order | • **Fan-out risk**: 112,650 items map to 98,666 unique orders. <br>• 510 orders contain items spanning multiple seller states, and 727 span multiple categories. | • Calculate rates by aggregating to the distinct **order** grain to avoid counting single orders multiple times. |
| **`sellers`** | One registered seller | • Contains seller state attributes. | • Join to `order_items` via `seller_id`. |
| **`products`** | One unique product | • 610 products have missing category names. | • Group these under an `'unknown'` category. |
| **`category_translation`** | One category translation mapping | • 2 Portuguese category names (`pc_gamer`, `portateis_cozinha...`) are missing from the English translation table. | • Use a fallback mechanism to preserve original Portuguese names if the English translation is missing. |

---

#### **2. Required Analytical Decisions (Aligned with Plan)**

*   **Decision 1: Which two dates to compare & What counts as "late"? (The Midnight Precision Issue)**
    *   **The Issue:** `order_estimated_delivery_date` lacks hours/minutes (always midnight), whereas `order_delivered_customer_date` has full timestamp precision. Comparing them directly (`actual > estimated`) would wrongly flag a delivery arriving at 14:00 on the estimated delivery date as "late", when it was actually delivered on time on that day.
    *   **The Decision:** We will compare `order_delivered_customer_date` against `order_estimated_delivery_date`. An order is defined as **late** only if it is delivered on a **calendar day strictly after** the estimated date. Delivered on the estimated day = on time. Mathematically, we compare the date parts:
        $$\text{Date}(\text{order\_delivered\_customer\_date}) > \text{Date}(\text{order\_estimated\_delivery\_date})$$

*   **Decision 2: How will we treat orders that were not delivered?**
    *   **The Decision:** We will exclude orders from the on-time delivery metric if they do not have a valid `order_delivered_customer_date` (or if their status is not `delivered`), since delivery performance cannot be evaluated. The 8 delivered orders with no date and 6 canceled orders with a date will also be left out, leaving an expected set of **96,470** delivered orders for calculation.

*   **Decision 3: Which month should we use?**
    *   **The Decision:** We will group and analyze trends using the **Purchase Month** (extracted from `order_purchase_timestamp`), as it reflects when the customer placed the order and is known for 100% of the dataset.

## 2. Plan


### **Analytical Framework and Design Decisions**

| Question | Design Choice | Rationale |
| :--- | :--- | :--- |
| **Which two dates?** | `order_delivered_customer_date` (actual customer arrival) vs. `order_estimated_delivery_date` (the operational promise date) | The core business metric is customer-centric arrival. Carrier dispatch dates only track fulfillment handoffs, not delivery performance. |
| **What is late?** | Delivered on a **calendar day strictly after** the estimated date. Delivery on or before the estimated day = on time. | Estimated dates are recorded at midnight, representing 'delivery by the end of that day.' Comparing raw timestamps would create false late flags. |
| **How to handle undelivered orders?** | **Exclude from the on-time rate calculation** and analyze as a separate operational metric. We also exclude the 6 canceled orders with dates and 8 delivered orders with missing dates. | Delivery lateness and complete non-delivery are separate operational issues. Conflating them would distort both key performance indicators. |
| **Which month?** | **Purchase Month** | This represents the time the customer placed the order, which is fully populated and captures customer-facing demand patterns. |

---

### **Outputs and Granularities**

1.  **Output 1 (Month × Seller State):**
    *   **Granularity:** One row per unique purchase month and seller state.
    *   **Metrics:** Total count of delivered orders containing an item from that seller state, and the percentage delivered on time.
    *   **Note on Grain:** An order with items from sellers in multiple states will count once under each state. Consequently, the sum of state-level orders will slightly exceed the total number of unique orders.

2.  **Output 2 (Product Categories):**
    *   **Granularity:** One row per product category (using English translation, falling back to Portuguese or 'unknown').
    *   **Metrics:** Total delivered orders, count of late orders, late rate, and average days late for late deliveries.

**Expected Dataset Size:** Exactly 96,470 clean, delivered orders (96,478 total 'delivered' status orders minus the 8 anomalies missing customer delivery dates).

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

**Prompt:** "Keep orders with status 'delivered' and a delivery date. Add delivered_date (the delivery date with the time removed), is_late = delivered_date > estimated date, days_late = days between them, and purchase_month. Print the row count and the late rate."

**Expect:** 96,470 rows and a late rate of roughly 5–10%.

In [8]:
delivered = orders[(orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].notna()].copy()

delivered["delivered_date"] = delivered["order_delivered_customer_date"].dt.normalize()
delivered["is_late"] = delivered["delivered_date"] > delivered["order_estimated_delivery_date"]
delivered["days_late"] = (delivered["delivered_date"] - delivered["order_estimated_delivery_date"]).dt.days
delivered["purchase_month"] = delivered["order_purchase_timestamp"].dt.to_period("M").astype(str)

print(len(delivered), "delivered orders | late:", delivered["is_late"].sum(), f"({delivered['is_late'].mean():.1%})")

96470 delivered orders | late: 6534 (6.8%)


**Prompt:** "How many orders would be late if I compared the full delivery timestamp with the estimated date instead?"

**Expect:** a few more late orders.

In [9]:
late_by_timestamp = delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
print("late by timestamp:", late_by_timestamp.sum(), "| late by date:", delivered["is_late"].sum(),
      "| delivered ON the estimated day but called late by the timestamp rule:", (late_by_timestamp & ~delivered["is_late"]).sum())

late by timestamp: 7826 | late by date: 6534 | delivered ON the estimated day but called late by the timestamp rule: 1292


Saw: more than expected. The timestamp comparison would call 1,292 extra orders late, every one of them delivered on the promised day. That would raise the late rate from 6.8% to 8.1%. I keep the date rule.

Prompt: "Make one row per delivered order and seller state: join delivered orders to order_items and sellers, then drop duplicate (order_id, seller_state) pairs. Print the rows before and after removing duplicates."

Expect: item rows above 96,470 (fan-out), and order–state pairs a little above 96,470 (up to 510 more).

In [10]:
order_state = (
    delivered[["order_id", "purchase_month", "is_late", "days_late"]]
    .merge(items[["order_id", "seller_id"]], on="order_id", how="inner")
    .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
)
print("item rows after the join:", len(order_state))

order_state = order_state.drop_duplicates(["order_id", "seller_state"])
print("order-state rows:", len(order_state), "| distinct orders:", order_state["order_id"].nunique(),
      "| missing state:", order_state["seller_state"].isna().sum())

item rows after the join: 110189
order-state rows: 96986 | distinct orders: 96470 | missing state: 0


Saw: as expected. The join fans out to 110,189 item rows. After removing repeats there are 96,986 order–state rows for 96,470 orders. The 516 extra rows are orders with sellers in more than one state. Every delivered order has items, and every seller has a state.

Prompt: Group order_state by purchase_month and seller_state: count orders, and work out the on-time rate as 1 minus the mean of is_late, rounded to 3 decimal places. Show the first rows and the number of rows.

Expect: a few hundred rows (about 23 months × up to 22 states).

In [11]:
month_state = (
    order_state.groupby(["purchase_month", "seller_state"])
    .agg(orders=("order_id", "nunique"), late=("is_late", "sum"))
    .reset_index()
)
month_state["on_time_rate"] = (1 - month_state["late"] / month_state["orders"]).round(3)

print(len(month_state), "rows")
month_state.head(10)

342 rows


,purchase_month,seller_state,orders,late,on_time_rate
0,2016-09,PR,1,1,0.000
1,2016-10,BA,1,0,1.000
2,2016-10,DF,2,0,1.000
3,2016-10,ES,1,0,1.000
4,2016-10,MG,21,0,1.000
5,2016-10,MS,3,0,1.000
6,2016-10,PR,32,1,0.969
7,2016-10,RJ,37,0,1.000
8,2016-10,RS,7,0,1.000
9,2016-10,SC,11,0,1.000


Table 1. One row = one purchase month × one seller state: the number of delivered orders with at least one item from a seller in that state, the number late, and the share delivered on or before the estimated date.

Many cells are small (a single seller state in one month), so a summary at each level is easier to read

Prompt: "Show the on-time rate by purchase month only (distinct orders), and by seller state only, keeping states with at least 500 orders."

Expect: most months above 90% on time, with a few dips.

In [12]:
by_month = delivered.groupby("purchase_month").agg(orders=("order_id", "size"), on_time_rate=("is_late", lambda s: round(1 - s.mean(), 3)))
by_month

,orders,on_time_rate
purchase_month,,
2016-09,1,0.000
2016-10,265,0.992
2016-12,1,1.000
2017-01,750,0.971
2017-02,1653,0.970
2017-03,2546,0.954
2017-04,2303,0.934
2017-05,3545,0.970
2017-06,3135,0.970


In [13]:
by_state = order_state.groupby("seller_state").agg(orders=("order_id", "nunique"), on_time_rate=("is_late", lambda s: round(1 - s.mean(), 3)))
by_state[by_state["orders"] >= 500].sort_values("on_time_rate")

,orders,on_time_rate
seller_state,,
SP,68635,0.927
RJ,4227,0.929
PR,7512,0.946
DF,808,0.947
SC,3603,0.951
BA,550,0.951
MG,7733,0.952
RS,1962,0.969


Saw:

Usually above 95% on time, with clear dips in Nov 2017 (87.6%) (the Black Friday peak) and Feb–Mar 2018 (85.9% and 81.0%).
Edge months: 2016-09 has a single order, so its 0% means nothing.

States: among seller states with at least 500 orders, on-time rates are close, from 92.7% (SP) to 96.9% (RS). SP (São Paulo) has 68,635 of the orders, so it drives the overall figure. States with fewer orders aren't compared here.

Prompt: "Make one row per delivered order and category: join delivered orders to order_items, products and category_translation, use the English name (falling back to the Portuguese name, then 'unknown'), and drop duplicate (order_id, category) pairs. Then per category count orders, late orders, the late rate and the average days_late of late orders. Rank by late orders and show the top 10."

Expect: the biggest categories at the top (lots of orders, so lots of late ones).

In [14]:
order_cat = (
    delivered[["order_id", "is_late", "days_late"]]
    .merge(items[["order_id", "product_id"]], on="order_id", how="inner")
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(translation, on="product_category_name", how="left")
)
order_cat["category"] = (order_cat["product_category_name_english"]
                         .fillna(order_cat["product_category_name"])
                         .fillna("unknown"))
order_cat = order_cat.drop_duplicates(["order_id", "category"])

categories = order_cat.groupby("category").agg(
    orders=("order_id", "nunique"),
    late_orders=("is_late", "sum"),
    avg_days_late=("days_late", lambda s: round(s[s > 0].mean(), 1)),
)
categories["late_rate"] = (categories["late_orders"] / categories["orders"]).round(3)

top_late = categories.sort_values("late_orders", ascending=False).head(10)
top_late

,orders,late_orders,avg_days_late,late_rate
category,,,,
bed_bath_table,9272,689,11.0,0.074
health_beauty,8647,649,9.8,0.075
sports_leisure,7529,495,10.8,0.066
furniture_decor,6307,449,11.1,0.071
computers_accessories,6529,417,10.3,0.064
watches_gifts,5493,406,9.8,0.074
housewares,5743,308,10.6,0.054
telephony,4093,291,9.2,0.071
auto,3809,278,12.3,0.073


Table 2. One row = one product category: delivered orders containing it, how many were late, the late rate, and the average days late for the late ones.

Saw: as expected, the top 10 by number of late orders are mostly the biggest categories (bed_bath_table, health_beauty, sports_leisure...). Their late rates are close to the 6.8% average, so they top the list because they're big, not because they're unusually late. Ranking by rate (categories with at least 300 orders) gives a different and more useful picture:

In [15]:
categories[categories["orders"] >= 300].sort_values("late_rate", ascending=False).head(10)

,orders,late_orders,avg_days_late,late_rate
category,,,,
audio,348,41,8.4,0.118
home_confort,392,37,13.5,0.094
office_furniture,1254,101,13.2,0.081
baby,2809,226,11.5,0.080
electronics,2517,192,9.3,0.076
musical_instruments,611,46,12.4,0.075
health_beauty,8647,649,9.8,0.075
furniture_living_room,414,31,10.7,0.075
watches_gifts,5493,406,9.8,0.074


Saw: a different list. audio (11.8% late), home_confort (9.4%) and office_furniture (8.1%) are late more often than the 6.8% average. Operations should see both lists: the count shows where the most customers are affected, and the rate shows where something is going wrong.

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. If applicable, include at least one example where the AI was wrong or where the output needed to be corrected.*

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*